#Section 1: Project Setup and Data Preparation

This section imports the required libraries, creates the folder structure for the repository,
loads the raw diabetes dataset, performs basic cleaning, saves cleaned versions of the data, 
and generates initial outputs that will be used in later project sections.

In [19]:
#--------------------------------
#Imported required libraries
#--------------------------------

#Standard Library
import os

#Data handling
import pandas as pd
import numpy as np

#Visualization
import matplotlib.pyplot as plt
import seaborn as sns

#Statistics
from scipy import stats

#Machine learning
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression #(Or whatever model you think is best)
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

#Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

print("All libraries imported successfully")

All libraries imported successfully


In [20]:
# ----------------------------
# Create repository folder structure
# ----------------------------

data_folder = "data"
outputs_folder = "outputs"
cleaned_data_folder = os.path.join(outputs_folder, "cleaned_data")
figures_folder = os.path.join(outputs_folder, "figures")
tables_folder = os.path.join(outputs_folder, "tables")

try:
    os.makedirs(data_folder, exist_ok=True)
    os.makedirs(outputs_folder, exist_ok=True)
    os.makedirs(cleaned_data_folder, exist_ok=True)
    os.makedirs(figures_folder, exist_ok=True)
    os.makedirs(tables_folder, exist_ok=True)

    print("\nFolder structure is ready.")
    print("Data folder:", data_folder)
    print("Cleaned data folder:", cleaned_data_folder)
    print("Figures folder:", figures_folder)
    print("Tables folder:", tables_folder)

except Exception as e:
    print("ERROR creating folders:", e)


Folder structure is ready.
Data folder: data
Cleaned data folder: outputs\cleaned_data
Figures folder: outputs\figures
Tables folder: outputs\tables


In [21]:
# ----------------------------
# Load the main dataset with error handling
# ----------------------------

file_path = os.path.join(data_folder, "diabetes_012_health_indicators_BRFSS2015.csv")

try:
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Dataset not found at path: {file_path}")

    df = pd.read_csv(file_path)

    print("\nDataset loaded successfully.")
    print("Dataset shape:", df.shape)

except FileNotFoundError as e:
    print("ERROR:", e)
    print("Make sure the dataset is placed inside the 'data/' folder.")

except pd.errors.EmptyDataError:
    print("ERROR: The dataset file is empty.")

except pd.errors.ParserError:
    print("ERROR: There was a problem parsing the CSV file.")

except Exception as e:
    print("An unexpected error occurred while loading the dataset:", e)


Dataset loaded successfully.
Dataset shape: (253680, 22)


In [22]:
# ----------------------------
# Initial inspection
# ----------------------------

try:
    print("\nFirst 5 rows:")
    display(df.head())

    print("\nColumn names:")
    print(df.columns.tolist())

    print("\nData types:")
    display(df.dtypes)

    print("\nMissing values per column:")
    display(df.isnull().sum())

    print("\nNumber of duplicated rows:")
    print(df.duplicated().sum())

except NameError:
    print("ERROR: Dataset was not loaded, so inspection cannot be performed.")


First 5 rows:


,Diabetes_012,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,HvyAlcoholConsump,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0



Column names:
['Diabetes_012', 'HighBP', 'HighChol', 'CholCheck', 'BMI', 'Smoker', 'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth', 'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Education', 'Income']

Data types:


Diabetes_012            float64
HighBP                  float64
HighChol                float64
CholCheck               float64
BMI                     float64
Smoker                  float64
Stroke                  float64
HeartDiseaseorAttack    float64
PhysActivity            float64
Fruits                  float64
Veggies                 float64
HvyAlcoholConsump       float64
AnyHealthcare           float64
NoDocbcCost             float64
GenHlth                 float64
MentHlth                float64
PhysHlth                float64
DiffWalk                float64
Sex                     float64
Age                     float64
Education               float64
Income                  float64
dtype: object


Missing values per column:


Diabetes_012            0
HighBP                  0
HighChol                0
CholCheck               0
BMI                     0
Smoker                  0
Stroke                  0
HeartDiseaseorAttack    0
PhysActivity            0
Fruits                  0
Veggies                 0
HvyAlcoholConsump       0
AnyHealthcare           0
NoDocbcCost             0
GenHlth                 0
MentHlth                0
PhysHlth                0
DiffWalk                0
Sex                     0
Age                     0
Education               0
Income                  0
dtype: int64


Number of duplicated rows:
23899


In [23]:
# ----------------------------
# Rename columns for readability
# ----------------------------

rename_map = {
    "Diabetes_012": "DiabetesStatus",
    "HighBP": "HighBloodPressure",
    "HighChol": "HighCholesterol",
    "CholCheck": "CholesterolCheck",
    "BMI": "BMI",
    "Smoker": "Smoker",
    "Stroke": "Stroke",
    "HeartDiseaseorAttack": "HeartDiseaseOrAttack",
    "PhysActivity": "PhysicalActivity",
    "Fruits": "Fruits",
    "Veggies": "Veggies",
    "HvyAlcoholConsump": "HeavyAlcoholConsumption",
    "AnyHealthcare": "AnyHealthcare",
    "NoDocbcCost": "NoDoctorBecauseCost",
    "GenHlth": "GeneralHealth",
    "MentHlth": "MentalHealthDays",
    "PhysHlth": "PhysicalHealthDays",
    "DiffWalk": "DifficultyWalking",
    "Sex": "Sex",
    "Age": "Age",
    "Education": "Education",
    "Income": "Income"
}

try:
    df = df.rename(columns=rename_map)
    print("\nColumns renamed successfully.")
    print(df.columns.tolist())

except NameError:
    print("ERROR: Dataset was not loaded, so columns cannot be renamed.")


Columns renamed successfully.
['DiabetesStatus', 'HighBloodPressure', 'HighCholesterol', 'CholesterolCheck', 'BMI', 'Smoker', 'Stroke', 'HeartDiseaseOrAttack', 'PhysicalActivity', 'Fruits', 'Veggies', 'HeavyAlcoholConsumption', 'AnyHealthcare', 'NoDoctorBecauseCost', 'GeneralHealth', 'MentalHealthDays', 'PhysicalHealthDays', 'DifficultyWalking', 'Sex', 'Age', 'Education', 'Income']


In [24]:
# ----------------------------
# Basic cleaning
# ----------------------------

try:
    duplicates_before = df.duplicated().sum()
    print("\nDuplicate rows before removal:", duplicates_before)

    df = df.drop_duplicates()

    duplicates_after = df.duplicated().sum()
    print("Duplicate rows after removal:", duplicates_after)

    print("\nMissing values after cleaning:")
    display(df.isnull().sum())

    print("\nCleaned dataset shape:", df.shape)

except NameError:
    print("ERROR: Dataset was not loaded, so cleaning cannot be performed.")


Duplicate rows before removal: 23899
Duplicate rows after removal: 0

Missing values after cleaning:


DiabetesStatus             0
HighBloodPressure          0
HighCholesterol            0
CholesterolCheck           0
BMI                        0
Smoker                     0
Stroke                     0
HeartDiseaseOrAttack       0
PhysicalActivity           0
Fruits                     0
Veggies                    0
HeavyAlcoholConsumption    0
AnyHealthcare              0
NoDoctorBecauseCost        0
GeneralHealth              0
MentalHealthDays           0
PhysicalHealthDays         0
DifficultyWalking          0
Sex                        0
Age                        0
Education                  0
Income                     0
dtype: int64


Cleaned dataset shape: (229781, 22)


In [25]:
# ----------------------------
# Optional dataset check function
# ----------------------------

def check_dataframe(dataframe):
    print("\nChecking dataset...")
    print("Shape:", dataframe.shape)
    print("Total missing values:", dataframe.isnull().sum().sum())
    print("Duplicate rows:", dataframe.duplicated().sum())
    print("Check complete.")

try:
    check_dataframe(df)

except NameError:
    print("ERROR: Dataset was not loaded, so dataframe checks cannot be performed.")


Checking dataset...
Shape: (229781, 22)
Total missing values: 0
Duplicate rows: 0
Check complete.


In [26]:
# ----------------------------
# Baseline descriptive statistics
# ----------------------------

try:
    print("\nSummary statistics:")
    display(df.describe())

    print("\nDiabetes status counts:")
    display(df["DiabetesStatus"].value_counts().sort_index())

    print("\nMean BMI by diabetes status:")
    display(df.groupby("DiabetesStatus")["BMI"].mean())

    print("\nMean Physical Activity by diabetes status:")
    display(df.groupby("DiabetesStatus")["PhysicalActivity"].mean())

    print("\nMean High Blood Pressure by diabetes status:")
    display(df.groupby("DiabetesStatus")["HighBloodPressure"].mean())

    print("\nMean High Cholesterol by diabetes status:")
    display(df.groupby("DiabetesStatus")["HighCholesterol"].mean())

except NameError:
    print("ERROR: Dataset was not loaded, so descriptive statistics cannot be calculated.")

except KeyError as e:
    print("ERROR: A needed column was not found:", e)



Summary statistics:


,DiabetesStatus,HighBloodPressure,HighCholesterol,CholesterolCheck,BMI,Smoker,Stroke,HeartDiseaseOrAttack,PhysicalActivity,Fruits,Veggies,HeavyAlcoholConsumption,AnyHealthcare,NoDoctorBecauseCost,GeneralHealth,MentalHealthDays,PhysicalHealthDays,DifficultyWalking,Sex,Age,Education,Income
count,229781.000000,229781.000000,229781.000000,229781.000000,229781.00000,229781.000000,229781.000000,229781.000000,229781.000000,229781.000000,229781.000000,229781.000000,229781.000000,229781.000000,229781.000000,229781.000000,229781.000000,229781.000000,229781.000000,229781.000000,229781.000000,229781.000000
mean,0.325627,0.454441,0.441760,0.959535,28.68567,0.465661,0.044756,0.103216,0.733355,0.612966,0.794813,0.060710,0.946075,0.092810,2.601151,3.505373,4.675178,0.185507,0.439231,8.086582,4.980568,5.890383
std,0.724623,0.497921,0.496598,0.197047,6.78636,0.498821,0.206767,0.304241,0.442206,0.487073,0.403839,0.238798,0.225871,0.290167,1.064685,7.713725,9.046568,0.388709,0.496295,3.093809,0.992895,2.092477
min,0.000000,0.000000,0.000000,0.000000,12.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000
25%,0.000000,0.000000,0.000000,1.000000,24.00000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000,2.000000,0.000000,0.000000,0.000000,0.000000,6.000000,4.000000,5.000000
50%,0.000000,0.000000,0.000000,1.000000,27.00000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,0.000000,1.000000,0.000000,3.000000,0.000000,0.000000,0.000000,0.000000,8.000000,5.000000,6.000000
75%,0.000000,1.000000,1.000000,1.000000,32.00000,1.000000,0.000000,0.000000,1.000000,1.000000,1.000000,0.000000,1.000000,0.000000,3.000000,2.000000,4.000000,0.000000,1.000000,10.000000,6.000000,8.000000
max,2.000000,1.000000,1.000000,1.000000,98.00000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,5.000000,30.000000,30.000000,1.000000,1.000000,13.000000,6.000000,8.000000



Diabetes status counts:


DiabetesStatus
0.0    190055
1.0      4629
2.0     35097
Name: count, dtype: int64


Mean BMI by diabetes status:


...

DiabetesStatus
0.0    0.754055
1.0    0.678332
2.0    0.628515
Name: PhysicalActivity, dtype: float64


Mean High Blood Pressure by diabetes status:


DiabetesStatus
0.0    0.395175
1.0    0.629078
2.0    0.752344
Name: HighBloodPressure, dtype: float64


Mean High Cholesterol by diabetes status:


DiabetesStatus
0.0    0.395349
1.0    0.620868
2.0    0.669459
Name: HighCholesterol, dtype: float64

In [27]:
# ----------------------------
# Save the main cleaned dataset
# ----------------------------

cleaned_main_file = os.path.join(cleaned_data_folder, "diabetes_cleaned.csv")

try:
    df.to_csv(cleaned_main_file, index=False)
    print("\nMain cleaned dataset saved successfully.")
    print("Saved to:", cleaned_main_file)

except NameError:
    print("ERROR: Dataset was not loaded, so cleaned data could not be saved.")

except Exception as e:
    print("ERROR saving cleaned dataset:", e)


Main cleaned dataset saved successfully.
Saved to: outputs\cleaned_data\diabetes_cleaned.csv


In [28]:
# ----------------------------
# Save a baseline summary table
# ----------------------------

try:
    summary_table = df.groupby("DiabetesStatus")[["BMI", "PhysicalActivity", "HighBloodPressure", "HighCholesterol"]].mean()
    summary_table_file = os.path.join(tables_folder, "baseline_summary_by_diabetes_status.csv")

    summary_table.to_csv(summary_table_file)

    print("\nBaseline summary table saved successfully.")
    print("Saved to:", summary_table_file)
    display(summary_table)

except NameError:
    print("ERROR: Dataset was not loaded, so the summary table could not be created.")

except KeyError as e:
    print("ERROR: A needed column was not found:", e)


Baseline summary table saved successfully.
Saved to: outputs\tables\baseline_summary_by_diabetes_status.csv


,BMI,PhysicalActivity,HighBloodPressure,HighCholesterol
DiabetesStatus,,,,
0.0,28.030528,0.754055,0.395175,0.395349
1.0,30.726075,0.678332,0.629078,0.620868
2.0,31.964242,0.628515,0.752344,0.669459


In [29]:
# ----------------------------
# Final repository file check
# ----------------------------

try:
    print("\nFiles currently in cleaned_data folder:")
    print(os.listdir(cleaned_data_folder))

    print("\nFiles currently in figures folder:")
    print(os.listdir(figures_folder))

    print("\nFiles currently in tables folder:")
    print(os.listdir(tables_folder))

except Exception as e:
    print("ERROR checking output folders:", e)


Files currently in cleaned_data folder:
['diabetes_cleaned.csv']

Files currently in figures folder:
[]

Files currently in tables folder:
['baseline_summary_by_diabetes_status.csv']
